# 📈 Сравнение методов PEFT — анализ результатов

В этом ноутбуке мы визуализируем и анализируем результаты четырёх экспериментов:

| Метод | Суть |
|-------|------|
| **Full Fine-Tuning** | Все параметры обучаются — верхняя граница качества |
| **LoRA** | Низкоранговое разложение матриц внимания |
| **Adapter** | Bottleneck-слои внутри трансформерных блоков |
| **Prefix Tuning** | Обучаемые виртуальные токены |

**Главный вопрос:** можно ли обучать <1% параметров и получить результат не хуже полного файнтюнинга?

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('Библиотеки загружены')

## 1. Загрузка результатов

Каждый эксперимент сохранил `test_results.json` в свою папку. Загружаем все четыре.

In [ ]:
# Результаты экспериментов
results = {
    'LoRA':             '../experiments/lora/test_results.json',
    'Adapter':          '../experiments/adapter/test_results.json',
    'Prefix Tuning':    '../experiments/prefix_tuning/test_results.json',
    'Full Fine-Tuning': '../experiments/full_finetune/test_results.json',
}

data = {}
for method, path in results.items():
    with open(path, encoding='utf-8') as f:
        data[method] = json.load(f)

# Количество обучаемых параметров (из логов обучения)
trainable_params = {
    'LoRA':             592131,
    'Adapter':          1200000,
    'Prefix Tuning':    370947,
    'Full Fine-Tuning': 178447878,
}

print('Результаты загружены:')
for method, d in data.items():
    print(f'  {method:20}: accuracy={d["accuracy"]:.4f}, f1_weighted={d["f1_weighted"]:.4f}')

## 2. Сравнение основных метрик

Начнём с главного — accuracy и F1 на тестовой выборке по всем методам.

In [ ]:
# Порядок методов по убыванию F1
methods = ['Prefix Tuning', 'LoRA', 'Full Fine-Tuning', 'Adapter']
colors  = ['#e67e22', '#3498db', '#95a5a6', '#2ecc71']

accuracy   = [data[m]['accuracy']    for m in methods]
f1_weighted = [data[m]['f1_weighted'] for m in methods]
f1_macro   = [data[m]['f1_macro']    for m in methods]

x = np.arange(len(methods))
width = 0.25

fig, ax = plt.subplots(figsize=(13, 6))

bars1 = ax.bar(x - width, accuracy,    width, label='Accuracy',    color=colors, alpha=0.6, edgecolor='white')
bars2 = ax.bar(x,         f1_weighted, width, label='F1 Weighted', color=colors, alpha=0.85, edgecolor='white')
bars3 = ax.bar(x + width, f1_macro,    width, label='F1 Macro',    color=colors, alpha=1.0, edgecolor='white')

# Подписи значений
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.003,
            f'{bar.get_height():.3f}',
            ha='center', va='bottom', fontsize=8, rotation=90
        )

ax.set_xticks(x)
ax.set_xticklabels(methods, fontsize=11)
ax.set_ylabel('Значение метрики')
ax.set_ylim(0.5, 0.80)
ax.set_title('Сравнение методов PEFT на тестовой выборке RuSentiment', fontsize=13, pad=15)
ax.legend(loc='lower right')
ax.axhline(y=data['Full Fine-Tuning']['f1_weighted'], color='gray', linestyle='--', alpha=0.5, linewidth=1)
ax.text(3.6, data['Full Fine-Tuning']['f1_weighted'] + 0.002, 'baseline', fontsize=9, color='gray')

plt.tight_layout()
plt.savefig('../experiments/comparison_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

**Вывод:** Adapter Tuning показывает наилучший результат, обходя даже Full Fine-Tuning. LoRA совсем немного уступает baseline. Prefix Tuning заметно отстаёт от остальных методов.

## 3. Эффективность: качество vs количество параметров

Главная ценность PEFT — соотношение качества и количества обучаемых параметров. Визуализируем этот trade-off.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

method_colors = {
    'LoRA':             '#3498db',
    'Adapter':          '#2ecc71',
    'Prefix Tuning':    '#e67e22',
    'Full Fine-Tuning': '#95a5a6',
}

for method in methods:
    params = trainable_params[method]
    f1     = data[method]['f1_weighted']
    color  = method_colors[method]

    # Размер точки пропорционален количеству параметров (логарифмически)
    size = np.log10(params) * 80

    ax.scatter(params, f1, s=size * 3, color=color, zorder=5, edgecolors='white', linewidth=1.5)
    ax.annotate(
        method,
        (params, f1),
        textcoords='offset points',
        xytext=(12, 5),
        fontsize=11,
        color=color,
        fontweight='bold'
    )

ax.set_xscale('log')
ax.set_xlabel('Количество обучаемых параметров (лог. шкала)', fontsize=11)
ax.set_ylabel('F1 Weighted на тестовой выборке', fontsize=11)
ax.set_title('Эффективность PEFT методов: качество vs параметры', fontsize=13, pad=15)
ax.grid(True, alpha=0.3)

# Аннотация с процентами
for method in methods:
    params = trainable_params[method]
    f1     = data[method]['f1_weighted']
    pct    = params / trainable_params['Full Fine-Tuning'] * 100
    ax.annotate(
        f'{pct:.2f}%',
        (params, f1),
        textcoords='offset points',
        xytext=(12, -12),
        fontsize=9,
        color='gray'
    )

plt.tight_layout()
plt.savefig('../experiments/comparison_efficiency.png', dpi=150, bbox_inches='tight')
plt.show()

**Вывод:** LoRA и Adapter находятся в «сладкой зоне» — высокое качество при минимальных параметрах. Особенно показателен Adapter: обучая в 150 раз меньше параметров чем Full Fine-Tuning, он показывает лучший результат.

## 4. F1 по классам

Агрегированные метрики скрывают важные детали. Посмотрим как каждый метод справляется с отдельными классами.

In [ ]:
classes = ['negative', 'neutral', 'positive']
class_colors = {'negative': '#e74c3c', 'neutral': '#95a5a6', 'positive': '#2ecc71'}

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

for ax, cls in zip(axes, classes):
    f1_scores = [data[m]['f1_per_class'][cls] for m in methods]
    bars = ax.barh(
        methods, f1_scores,
        color=class_colors[cls], alpha=0.8,
        edgecolor='white', linewidth=1.5
    )
    for bar, val in zip(bars, f1_scores):
        ax.text(
            val + 0.005, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=10
        )
    ax.set_title(f'Класс: {cls}', fontsize=12, pad=10)
    ax.set_xlim(0, 0.95)
    ax.set_xlabel('F1 Score')
    ax.axvline(x=0.7, color='gray', linestyle='--', alpha=0.4)

plt.suptitle('F1 Score по классам тональности для каждого метода', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../experiments/comparison_per_class.png', dpi=150, bbox_inches='tight')
plt.show()

**Вывод:** Нейтральный класс предсказывается лучше всего у всех методов (F1 ~0.79–0.81) — вероятно из-за более чёткого лексического сигнала. Негативный класс — самый сложный (F1 0.41–0.62), особенно для Prefix Tuning. Это типичная картина для русскоязычных датасетов тональности.

## 5. Матрицы ошибок

Матрица ошибок показывает **куда именно** ошибается модель. Это важно для понимания паттернов ошибок.

In [ ]:
import numpy as np

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
class_labels = ['neg', 'neu', 'pos']

plot_order = ['Prefix Tuning', 'LoRA', 'Full Fine-Tuning', 'Adapter']

for ax, method in zip(axes, plot_order):
    cm = np.array(data[method]['confusion_matrix'])
    # Нормализуем по строкам (recall по каждому классу)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    sns.heatmap(
        cm_norm, annot=True, fmt='.2f',
        xticklabels=class_labels,
        yticklabels=class_labels,
        cmap='Blues', ax=ax,
        vmin=0, vmax=1,
        linewidths=0.5,
        cbar=False
    )
    f1 = data[method]['f1_weighted']
    ax.set_title(f'{method}\nF1={f1:.3f}', fontsize=11, pad=8)
    ax.set_xlabel('Предсказание')
    ax.set_ylabel('Истина')

plt.suptitle('Нормализованные матрицы ошибок (по строкам = recall)', fontsize=13, y=1.04)
plt.tight_layout()
plt.savefig('../experiments/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

**Вывод:** Главный паттерн ошибок одинаков для всех методов — негативные тексты путаются с позитивными и наоборот. Нейтральный класс предсказывается устойчиво. Prefix Tuning сильнее всего путает негативные тексты с нейтральными.

## 6. Итоговая сводка

In [ ]:
print('=' * 65)
print('  ИТОГОВЫЕ РЕЗУЛЬТАТЫ СРАВНЕНИЯ МЕТОДОВ PEFT')
print('  Датасет: RuSentiment | Модель: rubert-base-cased')
print('=' * 65)
print(f'{"Метод":20} {"Accuracy":>10} {"F1 W":>8} {"Params":>12} {"Train%":>8}')
print('-' * 65)

sorted_methods = sorted(methods, key=lambda m: data[m]['f1_weighted'], reverse=True)
for method in sorted_methods:
    d      = data[method]
    params = trainable_params[method]
    pct    = params / trainable_params['Full Fine-Tuning'] * 100
    marker = ' <-- winner' if method == sorted_methods[0] else ''
    print(f'{method:20} {d["accuracy"]:>10.4f} {d["f1_weighted"]:>8.4f} {params:>12,} {pct:>7.2f}%{marker}')

print()
print('Главные выводы:')
print('  1. Adapter Tuning (0.67% параметров) превзошёл Full Fine-Tuning')
print('  2. LoRA (0.33% параметров) потерял лишь ~1.7% F1 vs baseline')
print('  3. Prefix Tuning показал наихудший результат на данной задаче')
print('  4. PEFT методы практически не уступают полному файнтюнингу')
print('     при многократном сокращении вычислительных затрат')
print('=' * 65)